# Bài 1  · Vector và độ tương đồng cosine
PGS.TS. Nguyễn Văn Hậu

Mục tiêu: đọc dữ liệu nguồn, dùng cùng schema, tính và diễn giải có điều kiện. Các điểm ở đây đo nhãn metadata; chưa đánh giá sở thích người dùng. Mở notebook trong thư mục Bai01. Cài `numpy`, `scikit-learn` và `jupyter` theo requirements.txt nếu môi trường chưa có.


In [ ]:
from pathlib import Path
import sys, math
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "source/cosine_lab.py").exists():
    raise RuntimeError("Mở notebook tại thư mục Bai01 chứa source/ và data/.")
sys.path.insert(0, str(ROOT / "source"))
from cosine_lab import load_movies, labels, encode, cosine_safe, AXES3, QUERY, CANDIDATES
from sklearn.metrics.pairwise import cosine_similarity
movies = load_movies()
print("Số phim:", len(movies))
print("Ba trục:", AXES3)
print("Dòng nguồn:", movies[QUERY])


## 1. Dữ liệu → vector
Mỗi vị trí trả lời về một nhãn. Dùng chung AXES3 cho mọi hàng; movieId là khóa tra cứu. Ma trận X có 5 hàng ứng viên × 3 cột nhãn, còn u là vector tham chiếu có 3 thành phần.


In [ ]:
u = encode(movies[QUERY], AXES3)
X = np.stack([encode(movies[mid], AXES3) for mid in CANDIDATES])
print("u =", u.astype(int))
print("X (5 × 3) =\n", X.astype(int))
for mid, v in zip(CANDIDATES, X):
    print(mid, movies[mid]["title"], "→", v.astype(int))
assert X.shape == (5, 3)


## 2. Một hàng tính có bước
Giữ căn chính xác ở phép tính tay; trong mã so kết quả trước làm tròn với sai số 10⁻¹². Zero là không xác định, khác điểm 0 có chuẩn dương.


In [ ]:
v = X[1]  # Men in Black
dot = float(u @ v)
nu, nv = float(np.linalg.norm(u)), float(np.linalg.norm(v))
print("dot =", dot, "; chuẩn u =", nu, "; chuẩn v =", nv)
hand = dot / math.sqrt(float(u @ u) * float(v @ v))
own = cosine_safe(u, v)
lib = float(cosine_similarity([u], [v])[0, 0])
print("Tính tay / NumPy / scikit-learn:", hand, own, lib)
assert abs(hand - own) < 1e-12 and abs(own - lib) < 1e-12
for mid, v in zip(CANDIDATES, X):
    print(movies[mid]["title"], "dot", int(u @ v), "cos", format(cosine_safe(u, v), ".4f"))
assert sum(int(u @ v) for v in X) == 7


## 3. Điều kiện và phản ví dụ
Các mảng cùng kích thước chưa đảm bảo cùng nghĩa trục. Hàm biết kích thước và số hữu hạn; người gọi phải dùng encoder chung. scikit-learn trả 0 ở vector 0 theo quy ước triển khai, trong khi hàm của bài giữ None.


In [ ]:
print("Toy Story:", cosine_safe(u, [0, 1, 0]))
print("Vector 0:", cosine_safe(u, [0, 0, 0]))
print("Quy ước thư viện:", cosine_similarity([u], [[0, 0, 0]])[0, 0])
assert cosine_safe(u, [0, 1, 0]) == 0
assert cosine_safe(u, [0, 0, 0]) is None
assert abs(cosine_safe(3*u, X[1]) - cosine_safe(u, X[1])) < 1e-12
for a, b in [([1, 0], [0, 1]), ([1, 0, 1], [0, 1, 1]), ([1], [1]), ([1, 0], [1, 1])]:
    print(a, b, "→", cosine_safe(a, b))


## 4. Giữ phim, thay biểu diễn
Lược đồ đầy đủ có 19 nhãn metadata, gồm IMAX. Ký hiệu `(no genres listed)` chỉ tình trạng thiếu nhãn, không tạo một tọa độ thể loại. Giao tập nhãn cung cấp một cách kiểm độc lập cho vector multi-hot.


In [ ]:
axes_full = sorted(set().union(*(labels(row) for row in movies.values())))
uf = encode(movies[QUERY], axes_full)
print("19 nhãn:", axes_full)
for mid in CANDIDATES:
    vf = encode(movies[mid], axes_full)
    a, b = labels(movies[QUERY]), labels(movies[mid])
    independent = len(a & b) / math.sqrt(len(a) * len(b))
    full = cosine_safe(uf, vf)
    assert abs(full - independent) < 1e-12
    print(movies[mid]["title"], "3 nhãn", round(cosine_safe(u, encode(movies[mid], AXES3)), 4), "19 nhãn", round(full, 4))
print("Casablanca: 3 nhãn", cosine_safe(u, encode(movies[912], AXES3)), "đủ nhãn", cosine_safe(uf, encode(movies[912], axes_full)))


## 5. Vận dụng và tự chấm
Trước khi chạy ô đáp án, tự tính cho v=(0;1;1), giải thích giới hạn kết luận, xét 3u và v=0. Rubric: mã hóa/trục 2 điểm, phép tính 3, diễn giải/hệ số dương 3, vector 0 2. Bài tập: chọn hai movieId khác; lập bảng 3 nhãn/19 nhãn, một hàng tính tay và nhận xét ≤150 từ.


In [ ]:
v = np.array([0., 1., 1.])
print("dot", u @ v, "; chuẩn", np.linalg.norm(u), np.linalg.norm(v))
print("cos", cosine_safe(u, v), "; 3u", cosine_safe(3*u, v), "; zero", cosine_safe(u, np.zeros(3)))
assert abs(cosine_safe(u, v) - .5) < 1e-12
from cosine_lab import run
report = run()
print("Kiểm toàn kho:", report["candidate_pool"], "ứng viên;", report["exact_equal_label_count"], "phim khác cùng tập nhãn")


Nguồn: [GroupLens](https://grouplens.org/datasets/movielens/latest/), bản movies.csv đi kèm có SHA-256; [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html). Chạy mã xác nhận phép tính, chưa xác nhận tính hữu ích của một hệ gợi ý.
